## 量子埋め込みカーネルを使用した分類器の学習

ここでは、量子コンピュータを用いて、機械学習のひとつであるSVM(Support Vector Machine)で使用するカーネル関数を量子的に構成・学習し、そのカーネルを使って分類を行います。

本サンプルはABCI-Q上で動作します。ABCI-Q上のPrefect環境の導入とワークフローの動作方法はUser Guideの[Prefectマニュアル](https://g-quat-abciq.github.io/abciq-docs/ja/apps/prefect_install/)をご覧ください。

本サンプルは「[Training quantum embedding kernels for classification](https://covalent.readthedocs.io/en/latest/tutorials/1_QuantumMachineLearning/quantum_embedding_kernel/source.html)」のCovalentのチュートリアルを元に、quri-parts-oqtopusを用いるように変更しPrefectに対応したものです。

本サンプルの実装はSVMの学習を１回のみ行います。
学習の最適化による分類器の性能の向上の確認など、さらに先の実装の詳細は参照元のチュートリアルをご覧ください。

### 1. 必要なライブラリのimport

In [ ]:
from prefect import flow, task

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.inspection import DecisionBoundaryDisplay
from quri_parts.circuit import QuantumCircuit
from quri_parts.circuit.inverse import inverse_circuit
from quri_parts_oqtopus.backend import OqtopusConfig, OqtopusSamplingBackend


np.random.seed(2022)

### 2. データの生成

この後のSVMによる分類学習に使用するデータセットを生成します。

`scikit-learn`の`make_blobs`関数を使って、２つの特徴量と２つのクラスを持つデータセットを作ります。
`get_data`では9個のクラスタをマトリクス状に配置し、それぞれに 2 個ずつのデータ点を持つ合計 18 点のサンプルデータを生成します。
`split_train_test`では、データを学習用 80%、テスト用 20% の割合に分割します。

In [ ]:
@task
def get_data(n_samples=18):
    centers = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3), (3, 1), (3, 2), (3, 3)]
    X, y = make_blobs(
        n_samples=n_samples, n_features=2, centers=centers, cluster_std=0.1, shuffle=False
    )
    # rescale labels to be -1, 1
    mapping = {0: -1, 1: 1, 2: -1, 3: 1, 4: -1, 5: 1, 6: -1, 7: 1, 8: -1}
    y = np.array([mapping[i] for i in y])
    X = X.astype(np.float32)
    y = y.astype(int)
    return X, y


@task
def split_train_test_data(X, y, test_size=0.2):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=3)
    return X_train, X_test, y_train, y_test

X, y = get_data()
X_train, X_test, y_train, y_test = split_train_test_data(X, y)

# 生成したデータセットをグラフにプロットする
cm = plt.cm.RdBu
cm_bright = ListedColormap(["#FF0000", "#0000FF"])
fig, ax = plt.subplots(1, 1, figsize=(6, 4), facecolor="w")
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright)
ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, marker="$\u25EF$", cmap=cm_bright)
ax.set_xlabel("First feature")
ax.set_ylabel("Second feature")
plt.tight_layout()
plt.show()

### 3.　量子カーネルの作成

繰り返しのlayerを持つパラメータ化回路(ansats)を設計し、それを用いて量子埋め込みカーネルを構築します。
各layerではゲートの回転角を用いて、２つの特徴量を5量子ビットに埋め込みます。

量子カーネルの実行では量子回路をquri-parts-oqtopusを使って実行します。

In [ ]:
def add_crz(circuit: QuantumCircuit, control: int, target: int, theta: float):
    """
    CRZ(theta) を CNOT と RZ に分解

    CRZ(theta)
      = RZ(theta/2)
        CX
        RZ(-theta/2)
        CX
    """
    circuit.add_RZ_gate(target, theta / 2)
    circuit.add_CNOT_gate(control, target)
    circuit.add_RZ_gate(target, -theta / 2)
    circuit.add_CNOT_gate(control, target)

@task
def layer(x, params, n_qubits, i0=0, inc=1):
    circuit = QuantumCircuit(n_qubits)

    i = i0

    # Hadamard + RZ(x) + RY(param)
    for q in range(n_qubits):
        circuit.add_H_gate(q)
        circuit.add_RZ_gate(q, x[i % len(x)])
        circuit.add_RY_gate(q, params[0][q])
        i += inc

    # Ring CRZ
    for q in range(n_qubits):
        add_crz(
            circuit,
            control=q,
            target=(q + 1) % n_qubits,
            theta=params[1][q],
        )

    return circuit

@task
def ansatz(x, params):
    n_layers = params.shape[0]
    n_qubits = params.shape[2]
    circuit = QuantumCircuit(n_qubits)
    for j in range(n_layers):
        sub = layer(
            x,
            params[j],
            n_qubits,
            i0=j * n_qubits
        )
        circuit.extend(sub)

    return circuit

backend = OqtopusSamplingBackend(OqtopusConfig.from_file("oqtopus-dev"))

@task
def kernel_circuit(params):

    def circuit(x1, x2, params):
        circuit1 = ansatz(x1, params)
        circuit2 = ansatz(x2, params)
        total = circuit1 + inverse_circuit(circuit2)

        shots = 1000
        job = backend.sample(
            total,
            device_id="qiskit_sim",
            shots=shots,
        )
        counts = job.result().counts
        print(f"x1={x1},x2={x2}: {counts}")
        n_states = 2 ** total.qubit_count
        probs = np.zeros(n_states)

        for state, count in counts.items():
            probs[state] = count / shots

        return probs

    return circuit

@task
def kernel(x1, x2, params):
    return kernel_circuit(params)(x1, x2, params)[0]

### 4. SVMの学習プログラムを構成する

量子カーネルを用いてSVM(Support Vector Machine)分類器を構築します。

`svm`では`scikit-learn`のSVCに全データセットのカーネルマトリクス(`kernel_matrix`)を与えています。
`kernel_matrix`の各要素は、量子カーネル実行を実行することで得られます。

最終的にワークフローの`init_qsvm`で、ランダムに生成した量子特徴を使って量子回路を組み、
１回だけSVMを学習し精度を測定しています。returnでは学習/テストの精度を返しています。

In [ ]:
def kernel_matrix(X1, X2, params):
    K = np.zeros((len(X1), len(X2)))
    for i in range(len(X1)):
        for j in range(len(X2)):
            K[i, j] = kernel(
                X1[i],
                X2[j],
                params
            )

    return K

@task
def init_params(num_qubits, num_layers):
    return np.random.uniform(0, 2*np.pi, (num_layers, 2, num_qubits))


@task
def svm(X, y, params):
    svm = SVC(kernel=lambda x1, x2: kernel_matrix(x1, x2, params))
    svm.fit(X, y)
    return svm

@task
def calc_accuracy(classifier, X, y_true):
    return 1 - np.count_nonzero(classifier.predict(X) - y_true) / len(y_true)

@flow(log_prints=True)
def init_qsvm(n_samples=18, num_wires=5, num_layers=3):
    X, y = get_data(n_samples)
    X_train, X_test, y_train, y_test = split_train_test_data(X, y)
    params = init_params(num_wires, num_layers)
    classifier = svm(X_train, y_train, params)
    train_acc = calc_accuracy(classifier=classifier, X=X_train, y_true=y_train)
    test_acc = calc_accuracy(classifier=classifier, X=X_test, y_true=y_test)
    return classifier, X_train, y_train, X_test, y_test, train_acc, test_acc

### 5. Prefectワークフローの実行と結果取得

ワークフローを実行し、結果を取得して、学習/テストの精度を表示します。

In [ ]:
result = init_qsvm(18, 5, 3)

qsvm, X_train, y_train, X_test, y_test, train_acc, test_acc = result
print(f"Train accuracy: {train_acc}")
print(f"Test accuracy: {test_acc}")